In [56]:
pip install psutil numpy h5py

<a target="_blank" href="https://colab.research.google.com/github/ANDA-NI-2026/NI-Day3-Data-Formats/blob/main/01_homework/01_exercises.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Structured Scientific Data with HDF5: Design, Access, and Compression

HDF5 is one of the most widely used scientific data formats. Many neuroscience tools, analysis pipelines, and standardised formats rely on it internally. In this notebook, we explore how HDF5 works at a practical level using the h5py library, which provides a NumPy-friendly interface to HDF5 files.

Our goals are to:

- Understand how to store structured data and metadata

- Read data efficiently and selectively

- Explore how compression affects disk usage and performance

- Throughout, we focus on design decisions: how we organise data, how we represent it, and how those choices influence performance.

## Setup

### Import Libraries

In [57]:
import h5py
import numpy as np
import time

### Utility Functions

In [58]:
import os
import psutil

def _format_bytes(bytes: float, precision: int = 2) -> str:
    """
    Takes a time in seconds and returns a string (e.g. ) that is more human-readable.

    Looking to do this in a real project?  Some alternatives:
      - `humanfriendly`: https://pypi.org/project/humanfriendly/#getting-started
    """

    if bytes < 0:
        raise ValueError("bytes must be non-negative")

    units = [("KB", 1000), ("MB", 1_000_000), ("GB", 1_000_000_000), ("TB", 1_000_000_000_000)]

    for unit, scale in reversed(units):
        if bytes >= scale:
            value = bytes / scale
            return f"{value:.{precision}f} {unit}"
    else:
        return f"{bytes} B"


def _disk_read() -> float:
    return psutil.Process(os.getpid()).io_counters().read_bytes

def _disk_write() -> float:
    return psutil.Process(os.getpid()).io_counters().write_bytes


class utils:
    format_bytes = _format_bytes
    bytes_read = _disk_read
    bytes_written = _disk_write

---

## Writing Mixed-Type, Labelled Data and Metadata to HDF5 Files using h5py

HDF5 allows us to move beyond simple array storage. Instead of writing one array per file, we can:
  - Store multiple datasets in a single file
  - Organise data hierarchically using groups
  - Attach metadata directly to datasets
  - Combine numeric arrays with strings and other data types

This makes HDF5 suitable for structured scientific data, where measurements, metadata, and experimental parameters belong together.

In this section, we focus on:
  - Creating datasets
  - Organising them into groups
  - Adding metadata using attributes
  - Designing clear, interpretable file structures

The emphasis is not on complexity, but on clarity. We aim to create files that are both structured and understandable.

### Reference

| Code | Description |
| :-- | :-- |
| `with h5py.File('temp.h5', 'w') as f:`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`# write content in f` | Open an HDF5 file for writing *with* a context manager |
| `f = h5py.File('filename.h5', 'w')` | Open an h5py file object for writing *without* a context manager |
| `f.close()` | Closes the h5py file after writing *without* a context manager and releases the linked file back to the operating system.  |
| `f.create_dataset('temp', data=x)` | Write an array called 'temp' with the data in the numpy array `x` into the HDF5 file |
| `f.create_dataset('data/temp', data=x)` | Write an array called 'temp' in the folder called "data" with the data in the numpy array `x` into the HDF5 file |
| `f.attrs['name'] = 'Session 1'` | Set an attribute as metadata onto the root group of the HDF5 file -- this works like a normal Python dictionary |
| `f['x'].attrs['id'] = 'ABC'` | Set an attribute as metadata onto the 'x' node of the HDF file |



### Exercises

**Example**: Write an HDF5 File called `temp.h5` with the following schema:
```
root/
└── temp: uint16, 1000 x 1  (temperature measurements over time)
```

In [59]:
temp = np.random.randint(15, 22, size=(1000,1)).astype(np.uint16)


In [60]:
with h5py.File('temp.h5', 'w') as f:
    f.create_dataset('temp', data=temp)
    f['temp'].attrs.update({
        'description': 'Temperature measurements over time'
    })

**Exercise**: Open up this `temp.h5` file with the  [myHDF5 data browser](https://myhdf5.hdfgroup.org/) and explore the data in the GUI.  Does this look how you expected? It should look something like the image below.

![browser view](https://github.com/ANDA-NI-2026/NI-Day3-Data-Formats/blob/main/01_homework/img/h5_1.png?raw=1)

**Exercise**: Write an HDF5 File called `ephys.h5` with the following schema and descriptions:

```
root/
├── time: float32, 1 x 1000 (trial time, in seconds)
├── voltage: int16, 4 x 1000 (voltage measurements for each recording channel)
└── chan_names: S, 4 (channel names for each recording channel)
```

In [61]:
t = np.linspace(0, 3, 1000).astype(np.float32)
voltage = np.random.normal(1, 1, size=(4, 1000)).astype(np.float32)
chan_names = ['CH01', 'CH02', 'CH03', 'CH05']

In [62]:
with h5py.File('ephys.h5', 'w') as f:
    f.create_dataset('time', data=t)
    f['time'].attrs['description'] = 'Trial time, in seconds.'
    f.create_dataset('voltage', data=voltage)
    f['voltage'].attrs['description'] = 'Voltage measurements for each recording channel.'
    f.create_dataset('chan_names', data=chan_names)
    f['chan_names'].attrs['description'] = 'Channel names for each recording channel.'

**Exercise**: Open up your `ephys.h5` file with the  [myHDF5 data browser](https://myhdf5.hdfgroup.org/) and explore your data in the GUI.  Does this look how you expected?

![browser view](https://github.com/ANDA-NI-2026/NI-Day3-Data-Formats/blob/main/01_homework/img/h5_2.png?raw=1)

**Exercise**: Write an HDF5 File called `motion_tracking.h5` with the following schema
```
root/
├── session_date: str
├── subject_id: str
├── camera:
│   ├── black_noise_image: uint16, 640 x 640 x 3 (reference image taken with lights out)
│   ├── image_width: uint16
│   ├── image_height: uint16
│   ├── shutter_speed: uint16
│   └── aperture: float32
│
└── motion_tracking:
    ├── time: uint32 1 x 3000 (session time, in milliseconds)
    ├── rb_pos: float32  2 x 3 x 3000 (XYZ coordinates of the center of each tracked rigid body)
    ├── rb_rot: float32  2 x 3 x 3000 (XYZ Euler rotations of each tracked rigid body)
    ├── xyz_names: str 1 x 3 (The spatial coordinate names)
    └── rb_names: str 1 x 2 (The name of each rigid body)


```

In [63]:
session_date = '2024-04-22'
subject_id = 'AD11'
camera_black_noise_im = np.random.randint(0, 30, size=(640, 640, 3)).astype(np.uint16)
im_width = 640
im_height = 640
shutter_speed = 800
aperture = 2.8
motion_time = (np.arange(0, 1000, step=1/shutter_speed)[:3000] * 1000).astype(np.uint32)
rb_pos = np.random.random(size=(2, 3, 3000)).astype(np.float32)
rb_rot = np.random.random(size=(2, 3, 3000)).astype(np.float32)
xyz_names = ['X', 'Y', 'Z']
rb_names = ['head', 'tail_base']

In [64]:
with h5py.File('motion_tracking.h5', 'w') as f:

    f.create_dataset('session_date', data=session_date)
    f.create_dataset('subject_id', data=subject_id)

    f.create_dataset('camera/black_noise_image', data=camera_black_noise_im)
    f.create_dataset('camera/image_width',  data=np.uint16(im_width))
    f.create_dataset('camera/image_height', data=np.uint16(im_height))
    f.create_dataset('camera/shutter_speed',data=np.uint16(shutter_speed))
    f.create_dataset('camera/aperture',     data=np.float32(aperture))

    f.create_dataset('motion_tracking/time',      data=motion_time)
    f.create_dataset('motion_tracking/rb_pos',    data=rb_pos)
    f.create_dataset('motion_tracking/rb_rot',    data=rb_rot)
    f.create_dataset('motion_tracking/xyz_names', data=np.array(xyz_names, dtype='S'))
    f.create_dataset('motion_tracking/rb_names',  data=np.array(rb_names,  dtype='S'))

**Exercise**: Open up your `motion_tracking.h5` file with the  [myHDF5 data browser](https://myhdf5.hdfgroup.org/) and explore your data in the GUI.  Does this look how you expected?

![browser view](https://github.com/ANDA-NI-2026/NI-Day3-Data-Formats/blob/main/01_homework/img/h5_3.png?raw=1)

## Reading Data from HDF5 Files

One of the strengths of HDF5 is that we do not need to load an entire dataset into memory; without loading all the data, we can:
  - Inspect file structure
  - Read selected datasets
  - Access metadata
  - Load only slices of large arrays

In this section, we practise navigating HDF5 files and reading data efficiently. We treat HDF5 files as structured containers and learn how to explore them programmatically.

Our goal is to become comfortable:
  - Inspecting file contents
  - Accessing nested groups
  - Reading partial data
  - Retrieving metadata stored as attributes


### Reference

| **Code** | **Description** |
| :-- | :-- |
| `with h5py.File('temp.h5', 'r') as f:`<br>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`# read content in f` | Open an HDF5 file for reading *with* a context manager |
| `f = h5py.File('filename.h5', 'r')` | Open an h5py file object for reading *without* a context manager |
| `f.close()` | Closes the h5py file after reading *without* a context manager.  |
| `f.keys()` | See a list of datasets and groups at the root node |
| `f.attrs` | Get the dict-like attributes at the root node |
| `f.attrs['a']` | Get the 'a' attribute at the root node  |
| `f['x'][:]` | Read in the 'x' dataset as a numpy array |
| `f['x'][5:20]` | Read in a slice of the 'x' dataset as a numpy array |
| `f['x'].keys()` | See a list of datasets and groups at the 'x' node |
| `f['folder']['x']` | Get the 'x' dataset in the 'folder' group |
| `f['x'].attrs['key'] ` | Get attribute named 'key_name' in the 'x' dataset |
| `f['folder/x']` | (Alternative Syntax) Get the 'x' dataset in the 'folder' group |





### Exercises

Run cell below to create `ephys.h5` and `motion_tracking.h5` files if you didn't already create them in the previous section.

In [65]:
t = np.linspace(0, 3, 1000).astype(np.float32)
voltage = np.random.normal(1, 1, size=(4, 1000)).astype(np.float32)
chan_names = ['CH01', 'CH02', 'CH03', 'CH05']

with h5py.File('ephys.h5', 'w') as f:
    f.create_dataset('time', data=t)
    f['time'].attrs.update({
        'description': 'Trial time, in seconds.'
    })
    f.create_dataset('voltage', data=voltage)
    f['voltage'].attrs.update({
        'description': 'voltage measurements for each recording channel.'
    })
    f.create_dataset('chan_names', data=chan_names)
    f['chan_names'].attrs.update({
        'description': 'channel names for each recording channel.'
    })

session_date = '2024-04-22'
subject_id = 'AD11'
camera_black_noise_im = np.random.randint(0, 30, size=(640, 640, 3)).astype(np.uint16)
im_width = 640
im_height = 640
shutter_speed = 800
aperture = 2.8
motion_time = (np.arange(0, 1000, step=1/shutter_speed)[:3000] * 1000).astype(np.uint32)
rb_pos = np.random.random(size=(2, 3, 3000)).astype(np.float32)
rb_rot = np.random.random(size=(2, 3, 3000)).astype(np.float32)
xyz_names = ['X', 'Y', 'Z']
rb_names = ['head', 'tail_base']

with h5py.File('motion_tracking.h5', 'w') as f:
    f.attrs.update({
        'session_date': session_date,
        'subject_id': subject_id,
    })
    f.create_dataset('camera/black_noise_image', data=camera_black_noise_im)
    f['camera'].attrs.update({
        'image_width': im_width,
        'image_height': im_height,
        'shutter_speed': shutter_speed,
        'aperture': aperture,
    })

    f.create_dataset('motion_tracking/time', data=motion_time)
    f.create_dataset('motion_tracking/rb_pos', data=rb_pos)
    f.create_dataset('motion_tracking/rb_rot', data=rb_rot)
    f.create_dataset('motion_tracking/rb_names', data=rb_names)
    f.create_dataset('motion_tracking/xyz_names', data=xyz_names)




**Example**: From the temperature file, read in only the last 5 temperature measurements as a numpy array.

In [66]:
f = h5py.File('temp.h5')
temp = f['temp'][-5:, :]
f.close()
temp

array([[18],
       [20],
       [19],
       [19],
       [20]], dtype=uint16)

**Exercise**: From the `ephys.h5` file, read in the first 10 voltage measurements as a numpy array.

In [67]:
ephys = h5py.File('ephys.h5')
voltage = ephys['voltage'][:10, :]
ephys.close()
voltage

array([[ 1.1075362 ,  1.7099183 , -0.2095001 , ...,  0.34437305,
         0.25386247,  0.3797999 ],
       [ 1.0862209 , -0.15248922,  2.0476835 , ...,  0.53671515,
         1.7224921 ,  1.0246661 ],
       [ 0.2885566 ,  4.253383  ,  3.411665  , ...,  1.3045357 ,
         0.5641294 ,  1.3373933 ],
       [ 1.7558441 ,  0.77904844,  0.20504704, ...,  2.4142065 ,
        -0.19709226, -0.03135862]], dtype=float32)

**Exercise**: From the `motion_tracking.h5` file, get the all the XYZ positions of the first rigid body during the recording.

In [68]:
with h5py.File('motion_tracking.h5', 'r') as f:
    rb_pos = f['motion_tracking/rb_pos'][:]

first_rb_xyz = rb_pos[0, :, :]

**Exercise**: from the `ephys.h5` file, load the name of the second recording channel.  (*Note*: Because HDF5 stores byte arrays for strings, to get it as a single string, it can help to use the  `.asstr()` method)

In [69]:
with h5py.File('ephys.h5', 'r') as f:
    second_channel = f['chan_names'].asstr()[1]

print(second_channel)

CH02


**Example**: From the `ephys.h5` file, load the description of the voltage dataset

In [70]:
f = h5py.File('ephys.h5')
description_voltage = f['voltage'].attrs['description']
f.close()
description_voltage

'voltage measurements for each recording channel.'

**Exercise**: From the motion tracking file, get the camera's shutter speed during the session

In [71]:
with h5py.File('motion_tracking.h5', 'r') as f:
    shutter_speed = f['camera'].attrs['shutter_speed']

print(shutter_speed)

800


---

## Compression in HDF5 — Trading CPU for Disk


When we store large datasets in HDF5, we can choose whether to compress them. Compression reduces file size and can potentially allow for faster disk reads (because less data is transferred), but it introduces trade-offs such as increased CPU usage during compression and decompression.

In this section, we will:

* Create datasets with and without compression
* Compare file sizes
* Measure write time and read time
* Observe CPU versus wall time trade-offs

### Reference

| Code                    | Description                      |
| :-- | :-- |
| `h5py.File()`           | Open or create an HDF5 file      |
| `create_dataset()`      | Create a dataset inside the file |
| `compression="gzip"`    | Enable gzip compression          |
| `compression_opts=`     | Set gzip compression level       |
| `dataset[:]`            | Read entire dataset              |
| `utils.bytes_written()` | Measure disk writes              |
| `array.nbytes`          | Measure array size in RAM        |



### Exercises


**Exercise**: "Writing an Uncompressed Dataset".  Please run the code below, and look at the printed performance.  Does disk usage closely match RAM size?


In [ ]:
data = np.arange(30_000_000, dtype=np.int64)
print("Size in RAM:", utils.format_bytes(data.nbytes))

dr0 = utils.bytes_written()
t0_wall = time.perf_counter()
t0_cpu = time.process_time()

with h5py.File("data.h5", "w") as f:
    f.create_dataset("data", data=data)

t1_wall = time.perf_counter()
t1_cpu = time.process_time()
dr1 = utils.bytes_written()

print("Disk written:", utils.format_bytes(dr1 - dr0))
print("Wall time:", t1_wall - t0_wall)
print("CPU time:", t1_cpu - t0_cpu)



Size in RAM: 240.00 MB
Disk written: 240.01 MB
Wall time: 0.2731432680011494
CPU time: 0.27302949899999973


**Exercise**: "Writing Structured Data with Gzip Compression". Please run the code below, and look at the printed performance.  Does disk usage closely match RAM size?  What changed from the uncompressed case?

In [ ]:
data = np.arange(30_000_000, dtype=np.int64)
print("Size in RAM:", utils.format_bytes(data.nbytes))


dr0 = utils.bytes_written()
t0_wall = time.perf_counter()
t0_cpu = time.process_time()

with h5py.File("data.h5", "w") as f:
    f.create_dataset(
        "data",
        data=data,
        compression="gzip"
    )

t1_wall = time.perf_counter()
t1_cpu = time.process_time()
dr1 = utils.bytes_written()

print("Disk written:", utils.format_bytes(dr1 - dr0))
print("Wall time:", t1_wall - t0_wall)
print("CPU time:", t1_cpu - t0_cpu)

Size in RAM: 240.00 MB
Disk written: 45.54 MB
Wall time: 5.272747287999664
CPU time: 5.244093512


**Exercise**: "Writing Random Noise with Gzip Compression". Please run the code below, and look at the printed performance.  Does disk usage closely match RAM size?  What changed from the structured case?  Is it always worth it to compress data?

In [ ]:
data = np.random.rand(30_000, 1_000)
print("Size in RAM:", utils.format_bytes(data.nbytes))



dr0 = utils.bytes_written()
t0_wall = time.perf_counter()
t0_cpu = time.process_time()

with h5py.File("data.h5", "w") as f:
    f.create_dataset(
        "data",
        data=data,
        compression="gzip"
    )

t1_wall = time.perf_counter()
t1_cpu = time.process_time()
dr1 = utils.bytes_written()

print("Disk written:", utils.format_bytes(dr1 - dr0))
print("Wall time:", t1_wall - t0_wall)
print("CPU time:", t1_cpu - t0_cpu)

Size in RAM: 240.00 MB
Disk written: 226.75 MB
Wall time: 20.424173425002664
CPU time: 20.411430742999997
